# 章节实践

通过本章的系统学习，我们已经掌握了使用Ascend C开发融合算子的完整方法。为了巩固所学知识，现提供以下实践练习：

实现Ascend C融合算子Matmul+Sinh,算子命名为MatmulSinh，编写其kernel侧代码、host侧代码，并完成测试。
相关算法：  
<div style="width: 30%; float: left; clear: both; margin: 10px 0;">

$$
\text{MatmulSinh}(c) = \text{Sinh}\left(\text{Matmul}(a, b) + bias\right)
$$
</div>
<!-- 清除浮动，避免影响后续内容排版 -->
<div style="clear: both;"></div>

算子原型：
<div style="text-align: left; float: left;">
<table style="border-collapse: collapse; width: 100%; max-width: 550px; border: 1px solid #ddd;">
  <!-- 表头行：算子类型 -->
  <tr>
    <td colspan="1" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">算子类型 (OpType)</td>
    <td colspan="4" style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;text-align: center;" > MatmulSinh</td>
  </tr>
  
  <!-- 算子输入模块：纵向合并单元格 -->
  <tr>
    <!-- 算子输入 纵向合并3行（x行+y行+列名行） -->
    <td rowspan="4" style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输入</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">name</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">shape</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">data type</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd; font-weight: bold;">format</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">a</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(M, K)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
  <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">b</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(K, N)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr>
   <tr>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">bias</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">(N)</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">float</td>
    <td style="padding: 8px; border-bottom: 1px solid #ddd;">ND</td>
  </tr> 
  <!-- 算子输出模块：纵向合并单元格 -->

  <tr>
  <td  style="padding: 8px; border-right: 1px solid #ddd; font-weight: bold; background-color: #f5f5f5; vertical-align: middle;">算子输出</td>
    <td style="padding: 8px;">c</td>
    <td style="padding: 8px;">(M, N)</td>
    <td style="padding: 8px;">float</td>
    <td style="padding: 8px;">ND</td>
  </tr>
</table>
</div>
<div style="clear: both;"></div>  

要求：

- 算子命名为MatmulSinh。

- 完成MatmulSinh算子kernel侧核函数实现相关代码补齐。

- 完成MatmulSinh算子host侧Tiling函数相关代码补齐。

- 支持float类型输入输出。

- 支持M = 1024, N = 2048, K = 512的输入。

请开始你的实践，体验完整开发过程。

---

环境准备：

In [ ]:
!mkdir -p Sources/05.05

import os, subprocess
env = subprocess.check_output("bash -l -c 'source $ASCEND_TOOLKIT_HOME/set_env.sh && env'", shell=True, text=True)
for line in env.splitlines():
    if "=" in line: os.environ.__setitem__(*line.split("=", 1))
print("\n🎉 Environment initialization process completed successfully!")

算子工程创建工具msopgen准备

In [ ]:
# 下载工具仓代码
!git clone https://gitcode.com/Ascend/msopgen.git
#编译安装msopgen工具
!cd msopgen; python build.py;cd output;pip install mindstudio_opgen*.whl --force-reinstall;pip install mindstudio_opst*.whl --force-reinstall


--- 
### **算子原型json文件准备**
该部分不需要修改，直接执行即可。

In [ ]:
%%writefile Sources/05.05/cv_fused_op.json
[
    {
        "op": "MatmulSinh",
        "language": "cpp",
        "input_desc": [
            {
                "name": "a",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            },
            {
                "name": "b",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            },
            {
                "name": "bias",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            }
        ],
        "output_desc": [
            {
                "name": "c",
                "param_type": "required",
                "format": ["ND"],
                "type": ["float"]
            }
        ]
    }
]

---
### **算子工程创建**
该部分不需要修改，直接执行即可。

In [ ]:
# 清除已有的custom_op目录
!rm -rf Sources/05.05/custom_op

# 使用msopgen工具创建算子工程前设置使用开源仓编译的msopgen工具，不设置时默认使用CANN安装目录下的msopgen工具
!export PATH=/usr/local/bin:~/.local/bin:$PATH;msopgen gen -i Sources/05.05/cv_fused_op.json -c ai_core-ascend910b1 -lan cpp -out Sources/05.05/custom_op


---
### **host侧代码修改**
修改代码，完成Tiling实现。

In [ ]:
%%writefile Sources/05.05/custom_op/op_host/matmul_sinh.cpp
#include "../op_kernel/matmul_sinh_tiling.h"
#include "register/op_def_registry.h"
#include "tiling/platform/platform_ascendc.h"
#include "tiling/tiling_api.h"
using namespace matmul_tiling;

namespace optiling {
static ge::graphStatus TilingFunc(gert::TilingContext* context)
{
  MatmulSinhTilingData *tiling = context->GetTilingData<MatmulSinhTilingData>();
  const gert::StorageShape* x1_shape = context->GetInputShape(0);
  int32_t data_sz = 1;
  for (int i = 0; i < x1_shape->GetStorageShape().GetDimNum(); i++)
    data_sz *= x1_shape->GetStorageShape().GetDim(i);
  tiling->size = data_sz;
  context->SetBlockDim(8);
  size_t *currentWorkspace = context->GetWorkspaceSizes(1);
  currentWorkspace[0] = 0;
  return ge::GRAPH_SUCCESS;
}
}

namespace ge {
static ge::graphStatus InferShape(gert::InferShapeContext* context)
{
    const gert::Shape* x1_shape = context->GetInputShape(0);
    gert::Shape* y_shape = context->GetOutputShape(0);
    *y_shape = *x1_shape;
    return GRAPH_SUCCESS;
}
static ge::graphStatus InferDataType(gert::InferDataTypeContext *context)
{
    const auto inputDataType = context->GetInputDataType(0);
    context->SetOutputDataType(0, inputDataType);
    return ge::GRAPH_SUCCESS;
}
}

namespace ops {
class MatmulSinh : public OpDef {
public:
    explicit MatmulSinh(const char* name) : OpDef(name)
    {
        this->Input("a")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("b")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Input("bias")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->Output("c")
            .ParamType(REQUIRED)
            .DataType({ge::DT_FLOAT})
            .Format({ge::FORMAT_ND})
            .UnknownShapeFormat({ge::FORMAT_ND});
        this->SetInferShape(ge::InferShape).SetInferDataType(ge::InferDataType);
        this->AICore()
            .SetTiling(optiling::TilingFunc);
        this->AICore().AddConfig("ascend910b");
    }
};
OP_ADD(MatmulSinh);
}

---
### **Tiling结构体定义修改**
修改代码，完成Tiling结构体定义

In [ ]:
%%writefile  Sources/05.05/custom_op/op_kernel/matmul_sinh_tiling.h
#ifndef MATMUL_SINH_TILING_H
#define MATMUL_SINH_TILING_H
#include <cstdint>
#include "kernel_tiling/kernel_tiling.h"

struct MatmulSinhTilingData {
    uint32_t size;
};
#endif

---
### **Kernel实现修改**
修改代码，完成Kernel实现

In [ ]:
%%writefile Sources/05.05/custom_op/op_kernel/matmul_sinh.cpp
#include "kernel_operator.h"
#include "matmul_sinh_tiling.h"
#include "lib/matmul_intf.h"

using namespace matmul;

extern "C" __global__ __aicore__ void matmul_sinh(GM_ADDR a, GM_ADDR b, GM_ADDR bias, GM_ADDR c, GM_ADDR workspace, GM_ADDR tiling) {
    REGISTER_TILING_DEFAULT(MatmulSinhTilingData);
    GET_TILING_DATA(tilingData, tiling);
    // TODO: user kernel impl
}

---
### **调用代码修改**
这里默认提供了M = 1024, N = 2048, K = 512的输入用例，可以不修改进行基础功能测试。

In [ ]:
%%writefile Sources/05.05/aclnn_test.cpp
#include <algorithm>
#include <cstdint>
#include <cstdio>
#include <vector>

#include "aclnn/aclnn_base.h"
#include "aclnn/acl_meta.h"
#include "acl/acl_rt.h"
#include "aclnn_matmul_sinh.h"

#define CHECK_ACL(expr)                                                                                 \
    do {                                                                                                \
        auto __ret = (expr);                                                                            \
        int32_t __code = static_cast<int32_t>(__ret);                                                   \
        if (__code != 0) {                                                                              \
            fprintf(stderr, "[ERROR] %s failed at %s:%d, ret=%d\n", #expr, __FILE__, __LINE__, __code); \
        }                                                                                               \
    } while (0)

int32_t main(int32_t argc, char** argv)
{
    const int32_t deviceId = 0;
    aclrtStream stream = nullptr;
    CHECK_ACL(aclnnInit(nullptr));
    CHECK_ACL(aclrtSetDevice(deviceId));
    CHECK_ACL(aclrtCreateStream(&stream));

    // 定义MatmulSinh的张量形状
    int64_t M = 1024;
    int64_t N = 2048;
    int64_t K = 512;
    const std::vector<int64_t> shape_a = {M, K};    // 输入a形状
    const std::vector<int64_t> shape_b = {K, N};     // 输入b形状
    const std::vector<int64_t> shape_bias = {N};       // 输入bias形状
    const std::vector<int64_t> shape_output = {M, N};// 输出形状

    // 计算各张量元素数量和内存大小
    const int64_t elementCount_a = shape_a[0] * shape_a[1];
    const int64_t elementCount_b = shape_b[0] * shape_b[1];
    const int64_t elementCount_bias = shape_bias[0];
    const int64_t elementCount_output = shape_output[0] * shape_output[1];
    
    const size_t bufferSize_a = elementCount_a * sizeof(float);
    const size_t bufferSize_b = elementCount_b * sizeof(float);
    const size_t bufferSize_bias = elementCount_bias * sizeof(float);
    const size_t bufferSize_output = elementCount_output * sizeof(float);

    // 分配输入a的设备内存并创建张量
    void* inputADeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputADeviceMem, bufferSize_a, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputA = aclCreateTensor(shape_a.data(), shape_a.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_a.data(), shape_a.size(), inputADeviceMem);

    // 分配输入b的设备内存并创建张量
    void* inputBDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBDeviceMem, bufferSize_b, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputB = aclCreateTensor(shape_b.data(), shape_b.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_b.data(), shape_b.size(), inputBDeviceMem);

    // 分配输入bias的设备内存并创建张量
    void* inputBiasDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&inputBiasDeviceMem, bufferSize_bias, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* inputBias = aclCreateTensor(shape_bias.data(), shape_bias.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                           shape_bias.data(), shape_bias.size(), inputBiasDeviceMem);

    // 分配输出的设备内存并创建张量
    void* outputDeviceMem = nullptr;
    CHECK_ACL(aclrtMalloc(&outputDeviceMem, bufferSize_output, ACL_MEM_MALLOC_HUGE_FIRST));
    aclTensor* output = aclCreateTensor(shape_output.data(), shape_output.size(), ACL_FLOAT, nullptr, 0, ACL_FORMAT_ND,
                                        shape_output.data(), shape_output.size(), outputDeviceMem);

    // 准备主机测试数据
    std::vector<float> inputAHostData(elementCount_a, float(0.125));
    std::vector<float> inputBHostData(elementCount_b, float(0.0625));
    std::vector<float> inputBiasHostData(elementCount_bias, float(0.5));
    std::vector<float> outputHostData(elementCount_output, float(0.0));
    
    // 计算预期结果：Sinh(0.125*0.0625*512 + 0.5) = 45.003011151991785
    std::vector<float> goldenData(elementCount_output, float(45.003011151991785));

    // 主机数据拷贝到设备
    CHECK_ACL(aclrtMemcpy(inputADeviceMem, bufferSize_a, inputAHostData.data(),
                          bufferSize_a, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBDeviceMem, bufferSize_b, inputBHostData.data(),
                          bufferSize_b, ACL_MEMCPY_HOST_TO_DEVICE));
    CHECK_ACL(aclrtMemcpy(inputBiasDeviceMem, bufferSize_bias, inputBiasHostData.data(),
                          bufferSize_bias, ACL_MEMCPY_HOST_TO_DEVICE));

    // 获取MatmulSinh算子工作空间大小和执行器
    uint64_t workspaceSize = 0;
    aclOpExecutor* executor = nullptr;
    CHECK_ACL(aclnnMatmulSinhGetWorkspaceSize(inputA, inputB, inputBias, output, &workspaceSize, &executor));

    // 分配工作空间内存
    void* workspaceDeviceMem = nullptr;
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtMalloc(&workspaceDeviceMem, workspaceSize, ACL_MEM_MALLOC_HUGE_FIRST));
    }

    // 执行MatmulSinh算子
    CHECK_ACL(aclnnMatmulSinh(workspaceDeviceMem, workspaceSize, executor, stream));
    CHECK_ACL(aclrtSynchronizeStream(stream));

    // 设备结果拷贝回主机
    CHECK_ACL(aclrtMemcpy(outputHostData.data(), bufferSize_output, outputDeviceMem,
                          bufferSize_output, ACL_MEMCPY_DEVICE_TO_HOST));

    // 打印结果并验证
    printf("result is:\n");
    const int64_t previewCount = std::min<int64_t>(elementCount_output, 10);
    for (int64_t i = 0; i < previewCount; i++) { 
        printf("%.1f ", outputHostData[i]); 
    }
    printf("\ntest %s\n", std::equal(outputHostData.begin(), outputHostData.end(), goldenData.begin()) ? "pass" : "failed");

    // 释放资源
    aclDestroyTensor(inputA);
    aclDestroyTensor(inputB);
    aclDestroyTensor(inputBias);
    aclDestroyTensor(output);
    
    CHECK_ACL(aclrtFree(inputADeviceMem));
    CHECK_ACL(aclrtFree(inputBDeviceMem));
    CHECK_ACL(aclrtFree(inputBiasDeviceMem));
    CHECK_ACL(aclrtFree(outputDeviceMem));
    
    if (workspaceSize > 0) {
        CHECK_ACL(aclrtFree(workspaceDeviceMem));
    }
    
    CHECK_ACL(aclrtDestroyStream(stream));
    CHECK_ACL(aclrtResetDevice(deviceId));
    CHECK_ACL(aclnnFinalize());
    
    return 0;
}

修改完成后，部署算子运行测试

In [ ]:
# 编译部署修改后的算子
!cd Sources/05.05/custom_op;bash build.sh;./build_out/custom_opp*.run --install-path=${HOME}/

In [ ]:
# 部署算子后编译调用代码，每次修改用例都需要重新编译
!g++ -I$ASCEND_TOOLKIT_HOME/include -I${HOME}/vendors/customize/op_api/include -L$ASCEND_TOOLKIT_HOME/lib64 -L${HOME}/vendors/customize/op_api/lib Sources/05.05/aclnn_test.cpp -lcust_opapi -lnnopbase -lacl_rt -o Sources/05.05/execute_op;
# 设置自定义算子so路径并执行调用代码
!source ${HOME}/vendors/customize/bin/set_env.bash;./Sources/05.05/execute_op

用例运行成功后会有类似以下的输出：
```

result is:
45.0 45.0 45.0 45.0 45.0 45.0 45.0 45.0 45.0 45.0 
test pass
```
